# Tahap 3 — Deteksi Code Smell dengan Qwen2.5-Coder-7B-Instruct
**Proyek:** Evaluasi Kinerja LLM dalam Deteksi Code Smells · *(versi rapi, hasil tersimpan ke Google Drive)*

Notebook ini menjalankan **empat varian prompt** pada model Qwen2.5-Coder-7B (4-bit) dan menyimpan
hasil tiap varian sebagai CSV **langsung ke Google Drive**, sehingga **tidak hilang** walau runtime
Colab ter-reset. Setiap loop **dapat dilanjutkan** (resumable).

| Varian | Strategi | Berkas hasil |
|---|---|---|
| v1 | Zero-shot (definisi + ambang) | `qwen_predictions.csv` |
| v2 | Chain-of-Thought (hitung baris/parameter) | `qwen_cot_predictions.csv` |
| v3 | Seimbang (smell hitungan + desain) | `qwen_v3_predictions.csv` |
| v4 | Few-shot (contoh tiap kelas) opsional | `qwen_v4_predictions.csv` |

> **Aktifkan GPU dulu:** Runtime → Change runtime type → **T4 GPU**.
>
> **Punya CSV hasil lama?** Agar tak perlu menjalankan ulang model, taruh berkas CSV Anda ke folder
> Drive `code-smell-llm/results/qwen/` (lewat Google Drive di browser). Loop akan mendeteksinya sudah
> lengkap (463 baris) dan otomatis melewatinya.

## Langkah 1 — Pasang dependensi & cek GPU

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
print("Selesai memasang dependensi.")

Selesai memasang dependensi.


In [ ]:
import torch
print("CUDA tersedia :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("VRAM          : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print("PERINGATAN: GPU belum aktif. Runtime > Change runtime type > T4 GPU.")

CUDA tersedia : True
GPU           : Tesla T4
VRAM          : 15.6 GB


## Langkah 2 — Hubungkan Google Drive (penyimpanan permanen)
Jalankan sel ini, lalu ikuti tautan izin yang muncul. Semua hasil akan disimpan di folder
`MyDrive/code-smell-llm/results/qwen/` dan tetap ada walau Colab mati.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
RESULTS = "/content/drive/MyDrive/code-smell-llm/results/qwen"
os.makedirs(RESULTS, exist_ok=True)
print("Hasil akan disimpan permanen di:", RESULTS)
print("Isi folder saat ini:", os.listdir(RESULTS) or "(masih kosong)")

Mounted at /content/drive
Hasil akan disimpan permanen di: /content/drive/MyDrive/code-smell-llm/results/qwen
Isi folder saat ini: ['efficiency', 'qwen_v4_predictions.csv', 'qwen_justifications.csv', 'token_counts_qwen.csv', 'qwen_predictions.csv', 'qwen_cot_predictions.csv', 'qwen_v3_predictions.csv']


## Langkah 3 — Unggah & ekstrak dataset

In [ ]:
from google.colab import files
import zipfile, os, glob
up = files.upload()                      # pilih dataset.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall("data_in")
gt_paths = glob.glob("data_in/**/ground_truth.csv", recursive=True)
assert gt_paths, "ground_truth.csv tidak ditemukan di dalam zip!"
DATASET_ROOT = os.path.dirname(gt_paths[0])
print("DATASET_ROOT =", DATASET_ROOT)

Saving dataset.zip to dataset.zip
DATASET_ROOT = data_in


In [ ]:
import pandas as pd
gt = pd.read_csv(os.path.join(DATASET_ROOT, "ground_truth.csv"))
gt["id"] = gt["id"].astype(str)
print("Total sampel:", len(gt))
print(pd.crosstab(gt["label"], gt["language"]))

def read_code(row):
    cp = str(row["code_path"]).replace("\\", "/")   # normalisasi pemisah path Windows -> Linux
    for p in [os.path.join(DATASET_ROOT, cp),
              os.path.join(DATASET_ROOT, cp.replace("dataset/", "")),
              os.path.join("data_in", cp)]:
        if os.path.exists(p):
            return open(p, encoding="utf-8", errors="replace").read()
    raise FileNotFoundError(cp)

# uji baca 1 berkas
print("\nContoh baca:", gt.iloc[0]["id"], "->", len(read_code(gt.iloc[0])), "karakter")

Total sampel: 463
language             java  python
label                            
data_class             42      24
feature_envy           24      39
god_class              42      40
long_method            42      42
long_parameter_list    42      42
non_smells             42      42

Contoh baca: java_data_class_001 -> 8441 karakter


## Langkah 4 — Muat model Qwen2.5-Coder (4-bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print("Model siap:", MODEL)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model siap: Qwen/Qwen2.5-Coder-7B-Instruct


## Langkah 5 — Definisi kelas, prompt dasar, dan fungsi penjalan
Sel ini memuat daftar kelas, definisi+ambang (dipakai semua varian), dan fungsi `run_loop` yang
menjalankan satu varian lalu menyimpan hasilnya ke Drive secara resumable.

In [ ]:
CLASSES = ["long_method", "god_class", "feature_envy",
           "data_class", "long_parameter_list", "non_smells"]

SYSTEM = ("You are an expert software engineer specialized in detecting code smells. "
          "You classify a single code unit into exactly ONE category and reply with strict JSON only.")

DEFINISI = """You are given ONE code unit (a single method/function, or a single class).
Classify it into EXACTLY ONE of the following six categories:

1. long_method        - a single method/function that is too long: MORE THAN 50 lines of code,
                        mixing too many responsibilities.
2. god_class          - a class that centralizes too much functionality (too many methods/fields,
                        low cohesion), violating the Single Responsibility Principle.
3. feature_envy       - a method that uses the data or methods of ANOTHER class more than those
                        of its own class.
4. data_class         - a class that only holds data (fields with getters/setters) and contains
                        little or no meaningful behavior.
5. long_parameter_list- a method/function that declares too many parameters: MORE THAN 7 parameters.
6. non_smells         - clean code that exhibits NONE of the smells above.

Rules:
- The unit contains AT MOST ONE smell. Choose exactly one label.
- If none of the five smells clearly applies, choose "non_smells".
- Base your judgment on the thresholds given (50 lines, 7 parameters) where relevant.
- Reply with STRICT JSON ONLY, no explanation, in this exact form: {"label": "<one_of_the_six>"}"""

def build_messages(code_text, language):
    user = (DEFINISI
            + f"\n\nLanguage: {language}\nCode unit:\n```{language}\n{code_text}\n```\n"
            + 'Now reply with JSON only, e.g. {"label": "long_method"}.')
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

In [ ]:
# Fungsi penjalan umum: menjalankan satu varian, menyimpan ke Drive, bisa dilanjutkan
import pandas as pd, os, time
from tqdm.auto import tqdm

def run_loop(classify_fn, parse_fn, out_name):
    out = os.path.join(RESULTS, out_name)
    rows = pd.read_csv(out).to_dict("records") if os.path.exists(out) else []
    done = {str(r["id"]) for r in rows}
    print(f"[{out_name}] sudah selesai: {len(done)} dari {len(gt)}")
    t0 = time.time()
    for _, r in tqdm(gt.iterrows(), total=len(gt), desc=out_name):
        if str(r["id"]) in done:
            continue
        try:
            raw = classify_fn(read_code(r), r["language"])
            lab, ok = parse_fn(raw)
        except Exception as e:
            raw, lab, ok = f"ERROR: {e}", "non_smells", False
        rows.append({"id": str(r["id"]), "language": r["language"], "true_label": r["label"],
                     "qwen_pred": lab, "parsed": ok, "qwen_raw": str(raw)[:400]})
        done.add(str(r["id"]))
        if len(rows) % 10 == 0:
            pd.DataFrame(rows).to_csv(out, index=False)
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f"[{out_name}] SELESAI {len(rows)} sampel dalam {(time.time()-t0)/60:.1f} menit -> {out}")

---
# Varian 1 — Zero-shot
Definisi + ambang eksplisit, model menjawab JSON langsung.

In [ ]:
import json, re, gc

MAX_CODE_CHARS = 8000   # batasi panjang kode -> cegah CUDA out of memory di T4

@torch.no_grad()
def classify_raw(code_text, language, max_new_tokens=64):
    code_text = code_text[:MAX_CODE_CHARS]
    def _run(ct, mlen):
        msgs = build_messages(ct, language)
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=mlen).to(model.device)
        try:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            del inputs
            gc.collect(); torch.cuda.empty_cache()
    try:
        return _run(code_text, 3072)
    except Exception as e:
        if "out of memory" not in str(e).lower():
            raise
        gc.collect(); torch.cuda.empty_cache()
        try:
            return _run(code_text[:2500], 2048)          # coba lagi, lebih pendek
        except Exception as e2:
            gc.collect(); torch.cuda.empty_cache()
            return f"ERROR_OOM: {e2}"

def parse_label(resp):
    m = re.search(r"\{[^{}]*\}", resp, re.S)
    if m:
        try:
            lab = str(json.loads(m.group(0)).get("label", "")).strip().lower()
            if lab in CLASSES:
                return lab, True
        except Exception:
            pass
    low = resp.lower()
    for c in CLASSES:
        if c in low:
            return c, True
    return "non_smells", False

In [ ]:
run_loop(classify_raw, parse_label, "qwen_predictions.csv")

[qwen_predictions.csv] sudah selesai: 463 dari 463


qwen_predictions.csv:   0%|          | 0/463 [00:00<?, ?it/s]

[qwen_predictions.csv] SELESAI 463 sampel dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/qwen/qwen_predictions.csv


# Varian 2 — Chain-of-Thought
Model menghitung baris & parameter dulu sebelum memutuskan.

In [ ]:
import json, re, gc

SYSTEM_COT = ("You are an expert software engineer specialized in detecting code smells. "
              "You reason step by step, then give a single final label as JSON.")

def build_messages_cot(code_text, language):
    user = (DEFINISI
        + f"\n\nLanguage: {language}\nCode unit:\n```{language}\n{code_text}\n```\n\n"
        + "Analyze step by step:\n"
        + "Step 1 - If it is a method/function: count the number of lines of code in its body, "
          "and count the number of parameters in its signature. State both numbers explicitly.\n"
        + "Step 2 - Check the smells:\n"
        + "   - long_method: is the line count > 50?\n"
        + "   - long_parameter_list: is the parameter count > 7?\n"
        + "   - god_class / data_class: judge only if it is a class.\n"
        + "   - feature_envy: examine the method body. If it PREDOMINANTLY calls methods or "
          "accesses fields of a SINGLE external object/class (often passed as a parameter), and "
          "makes little or no use of its own class members (this/self), it is likely feature_envy. "
          "Identify the object it uses most and estimate how many calls go to it versus to its own class.\n"
        + "Step 3 - Decide exactly ONE label (or non_smells if none clearly applies).\n\n"
        + 'After your reasoning, end with the final answer as JSON on its own line: {"label": "<one_of_the_six>"}')
    return [{"role": "system", "content": SYSTEM_COT},
            {"role": "user", "content": user}]

def parse_label_cot(resp):
    for m in reversed(re.findall(r"\{[^{}]*\}", resp, re.S)):   # JSON terakhir yang valid
        try:
            lab = str(json.loads(m).get("label", "")).strip().lower()
            if lab in CLASSES: return lab, True
        except Exception: pass
    low = resp.lower(); best_pos, best = -1, None              # cadangan: label terakhir disebut
    for c in CLASSES:
        i = low.rfind(c)
        if i > best_pos: best_pos, best = i, c
    return (best, True) if best else ("non_smells", False)

MAX_CODE_CHARS = 8000

@torch.no_grad()
def classify_cot(code_text, language, max_new_tokens=400):
    code_text = code_text[:MAX_CODE_CHARS]
    def _run(ct, mlen):
        text = tok.apply_chat_template(build_messages_cot(ct, language),
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=mlen).to(model.device)
        try:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            del inputs; gc.collect(); torch.cuda.empty_cache()
    try:
        return _run(code_text, 3072)
    except Exception as e:
        if "out of memory" not in str(e).lower(): raise
        gc.collect(); torch.cuda.empty_cache()
        try:    return _run(code_text[:2500], 2048)
        except Exception as e2:
            gc.collect(); torch.cuda.empty_cache()
            return f"ERROR_OOM: {e2}"

In [ ]:
run_loop(classify_cot, parse_label_cot, "qwen_cot_predictions.csv")

[qwen_cot_predictions.csv] sudah selesai: 463 dari 463


qwen_cot_predictions.csv:   0%|          | 0/463 [00:00<?, ?it/s]

[qwen_cot_predictions.csv] SELESAI 463 sampel dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/qwen/qwen_cot_predictions.csv


# Varian 3 — Seimbang (prompt utama)
Memisahkan jalur kelas vs method; smell desain dinilai kualitatif.

In [ ]:
import json, re, gc

SYSTEM_V3 = ("You are an expert software engineer specialized in detecting code smells. "
             "You reason step by step, then give a single final label as JSON.")

def build_messages_v3(code_text, language):
    user = (DEFINISI
        + f"\n\nLanguage: {language}\nCode unit:\n```{language}\n{code_text}\n```\n\n"
        + "Analyze step by step:\n\n"
        + "Step 1 - Decide whether this unit is a CLASS or a single METHOD/FUNCTION.\n\n"
        + "Step 2 - Apply the checks for that kind:\n"
        + "  If it is a METHOD/FUNCTION:\n"
        + "    (a) Count its lines of code -> if > 50, it is long_method.\n"
        + "    (b) Count its parameters -> if > 7, it is long_parameter_list.\n"
        + "    (c) feature_envy: if it predominantly calls methods/accesses fields of a SINGLE "
          "external object/class and barely uses its own (this/self), it is feature_envy.\n"
        + "  If it is a CLASS (judge these QUALITATIVELY, NOT by line count):\n"
        + "    (a) god_class: does it concentrate MANY responsibilities - many methods and fields, "
          "handling data AND logic AND coordination across several concerns, with low cohesion? "
          "A large class packed with diverse behavior is a god_class.\n"
        + "    (b) data_class: does it contain ALMOST NO real logic - only fields plus "
          "getters/setters/constructors and trivial accessors? A class that merely holds data.\n"
        + "    (c) Distinguish carefully: a god_class HAS substantial, varied behavior; a "
          "data_class has little or no behavior.\n\n"
        + "Step 3 - Choose exactly ONE label.\n"
        + "  - Do NOT pick non_smells merely because no numeric threshold was crossed: god_class "
          "and data_class are design smells judged by structure and responsibility, not by counts.\n"
        + "  - Pick non_smells ONLY if the code is genuinely clean and fits none of the five smells.\n\n"
        + 'After your reasoning, end with the final answer as JSON on its own line: {"label": "<one_of_the_six>"}')
    return [{"role": "system", "content": SYSTEM_V3},
            {"role": "user", "content": user}]

def parse_label_v3(resp):
    for m in reversed(re.findall(r"\{[^{}]*\}", resp, re.S)):
        try:
            lab = str(json.loads(m).get("label", "")).strip().lower()
            if lab in CLASSES: return lab, True
        except Exception: pass
    low = resp.lower(); best_pos, best = -1, None
    for c in CLASSES:
        i = low.rfind(c)
        if i > best_pos: best_pos, best = i, c
    return (best, True) if best else ("non_smells", False)

MAX_CODE_CHARS = 8000

@torch.no_grad()
def classify_v3(code_text, language, max_new_tokens=400):
    code_text = code_text[:MAX_CODE_CHARS]
    def _run(ct, mlen):
        text = tok.apply_chat_template(build_messages_v3(ct, language),
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=mlen).to(model.device)
        try:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            del inputs; gc.collect(); torch.cuda.empty_cache()
    try:
        return _run(code_text, 3072)
    except Exception as e:
        if "out of memory" not in str(e).lower(): raise
        gc.collect(); torch.cuda.empty_cache()
        try:    return _run(code_text[:2500], 2048)
        except Exception as e2:
            gc.collect(); torch.cuda.empty_cache()
            return f"ERROR_OOM: {e2}"

## Langkah 5b — Hitung panjang masukan Qwen (8.000 karakter & 3.072 token)
Untuk Subbab 3.8 dan Lampiran B tesis. Memakai **`build_messages_v3` asli** (prompt seimbang) dan tokenizer Qwen; **model tidak dimuat**.

Urutan menjalankan: sel Langkah 2 (Drive) → Langkah 3 (unggah dataset + `read_code`) → sel definisi `DEFINISI` (Langkah 5) → sel definisi Varian 3 → sel ini. Sel **Langkah 4 (muat model) dilewati**.

Catatan: di `classify_v3`, prompt dipotong dengan `tok(..., truncation=True, max_length=3072)`. Pemotongan ini membuang bagian **akhir** prompt (instruksi Step 3 dan perintah JSON), jadi penting diketahui berapa unit yang melewatinya.

In [ ]:
# === PANJANG MASUKAN QWEN v3: batas 8.000 karakter kode & 3.072 token prompt ===
import os, pandas as pd
from transformers import AutoTokenizer
from google.colab import files

if "tok" not in globals():                                   # hanya tokenizer, tanpa model
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")

MAX_CODE_CHARS, MAX_PROMPT_TOK = 8000, 3072
rows = []
for _, r in gt.iterrows():
    kode = read_code(r)
    teks = tok.apply_chat_template(build_messages_v3(kode[:MAX_CODE_CHARS], r["language"]),
                                   tokenize=False, add_generation_prompt=True)
    n_tok = len(tok(teks)["input_ids"])
    rows.append({"id": str(r["id"]), "language": r["language"], "label": r["label"],
                 "n_chars": len(kode), "terpotong_8000_karakter": len(kode) > MAX_CODE_CHARS,
                 "n_prompt_tokens": n_tok, "terpotong_3072_token": n_tok > MAX_PROMPT_TOK})
tq = pd.DataFrame(rows)

# Gabungkan dengan prediksi v3 untuk membandingkan ketepatan unit terpotong vs tidak
v3_path = os.path.join(RESULTS, "qwen_v3_predictions.csv")
if os.path.exists(v3_path):
    _v3 = pd.read_csv(v3_path); _v3["id"] = _v3["id"].astype(str)
    tq = tq.merge(_v3[["id", "qwen_pred"]], on="id", how="left")
    tq["benar"] = tq["qwen_pred"] == tq["label"]

out_tq = os.path.join(RESULTS, "token_counts_qwen.csv")
tq.to_csv(out_tq, index=False)

print(tq.groupby(["label", "language"])[["terpotong_8000_karakter", "terpotong_3072_token"]].sum())
print("\nTotal kode > 8.000 karakter : %d dari %d" % (tq["terpotong_8000_karakter"].sum(), len(tq)))
print("Total prompt > 3.072 token  : %d dari %d" % (tq["terpotong_3072_token"].sum(), len(tq)))
print("Statistik n_prompt_tokens   :", tq["n_prompt_tokens"].describe().round(0).to_dict())
if "benar" in tq:
    for kol in ["terpotong_8000_karakter", "terpotong_3072_token"]:
        print(f"\nAkurasi v3 menurut {kol}:")
        print(tq.groupby(kol)["benar"].agg(["mean", "count"]).round(3))
files.download(out_tq)


                              terpotong_8000_karakter  terpotong_3072_token
label               language                                               
data_class          java                            4                     0
                    python                          1                     0
feature_envy        java                            0                     0
                    python                          0                     0
god_class           java                           18                     0
                    python                         39                     0
long_method         java                            1                     0
                    python                          0                     0
long_parameter_list java                            0                     0
                    python                          3                     0
non_smells          java                            1                     0
            

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
run_loop(classify_v3, parse_label_v3, "qwen_v3_predictions.csv")

[qwen_v3_predictions.csv] sudah selesai: 463 dari 463


qwen_v3_predictions.csv:   0%|          | 0/463 [00:00<?, ?it/s]

[qwen_v3_predictions.csv] SELESAI 463 sampel dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/qwen/qwen_v3_predictions.csv


In [ ]:
# === TIMING QWEN v3 (efisiensi / RM3) ===
import time, os, pandas as pd, torch
from tqdm.auto import tqdm

EFF = os.path.join(RESULTS, "efficiency"); os.makedirs(EFF, exist_ok=True)
MAX_CODE_CHARS = 8000   # sama dengan pipeline Anda

@torch.no_grad()
def _time_generate_v3(code_text, language):
    msgs = build_messages_v3(code_text[:MAX_CODE_CHARS], language)   # prompt v3 Anda
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    torch.cuda.synchronize()                       # WAJIB: tunggu GPU sebelum mulai stopwatch
    t0 = time.perf_counter()
    _ = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()                       # WAJIB: tunggu GPU selesai
    return (time.perf_counter() - t0) * 1000.0

# Pemanasan 3 sampel (buang efek warmup CUDA pada pengukuran pertama)
for _, r in gt.head(3).iterrows():
    try: _time_generate_v3(read_code(r), r["language"])
    except Exception: pass
torch.cuda.empty_cache()

rows = []
for _, r in tqdm(gt.iterrows(), total=len(gt), desc="timing Qwen v3"):
    try:
        dt = _time_generate_v3(read_code(r), r["language"])
        rows.append({"id": str(r["id"]), "language": r["language"],
                     "detector": "Qwen v3", "time_ms": dt})
    except Exception as e:
        pass                                       # lewati sampel error (mis. OOM)
    if len(rows) % 20 == 0:
        torch.cuda.empty_cache()                   # berkala, BUKAN tiap sampel

df = pd.DataFrame(rows)
df.to_csv(os.path.join(EFF, "qwen_timing.csv"), index=False)
print("Qwen: %d entri | rata-rata %.0f ms | median %.0f ms"
      % (len(df), df.time_ms.mean(), df.time_ms.median()))

timing Qwen v3:   0%|          | 0/463 [00:00<?, ?it/s]

Qwen: 463 entri | rata-rata 2444 ms | median 1886 ms


# Varian 4 — Few-shot (opsional)
Menambah 6 contoh tulisan-tangan (bukan dari dataset, agar tanpa kebocoran data).

In [ ]:
import json, re, gc

SYSTEM_V4 = ("You are an expert software engineer specialized in detecting code smells. "
             "You learn from examples, reason briefly, then give a single final label as JSON.")

FEWSHOT = """Here are reference examples (one per category):

Example 1 (java):
void createUser(String name, String email, int age, String addr, String phone, String role, boolean active, String dept, String mgr) { save(name, email, age); }
Reasoning: 9 parameters > 7. -> {"label": "long_parameter_list"}

Example 2 (java):
class Point { private int x; private int y; public int getX(){return x;} public int getY(){return y;} public void setX(int v){x=v;} public void setY(int v){y=v;} }
Reasoning: only fields plus getters/setters, no real behavior. -> {"label": "data_class"}

Example 3 (java):
class AppManager { void addUser(){...} void deleteUser(){...} void processPayment(){...} void refund(){...} void sendEmail(){...} void generateReport(){...} void writeLog(){...} void loadConfig(){...} void backupDb(){...} /* many more, mixed concerns */ }
Reasoning: many methods spanning many unrelated concerns (users, payments, email, reports, config) -> low cohesion. -> {"label": "god_class"}

Example 4 (python):
def handle_request(req):
    # ~60 lines: parse input, validate, query db, transform, format response, handle errors, log
    ...
    return result
Reasoning: body well over 50 lines doing many different tasks. -> {"label": "long_method"}

Example 5 (java):
double price(Order order) { return order.getQty() * order.getUnitPrice() - order.getDiscount() + order.getTax(); }
Reasoning: uses Order's data repeatedly and almost nothing of its own class. -> {"label": "feature_envy"}

Example 6 (python):
def add(a, b):
    return a + b
Reasoning: small, clean, single clear purpose. -> {"label": "non_smells"}
"""

def build_messages_v4(code_text, language):
    user = (DEFINISI + "\n\n" + FEWSHOT
        + f"\nNow analyze THIS unit.\nLanguage: {language}\nCode unit:\n```{language}\n{code_text}\n```\n\n"
        + "Reason step by step:\n"
        + "Step 1 - Is it a CLASS or a METHOD/FUNCTION?\n"
        + "Step 2 - If METHOD: count lines (>50 = long_method), count parameters (>7 = "
          "long_parameter_list), check feature_envy (uses one other object far more than its own). "
          "If CLASS: judge god_class (many varied responsibilities) vs data_class (only data) "
          "QUALITATIVELY, not by line count.\n"
        + "Step 3 - Choose ONE label. Do NOT default to non_smells just because no numeric "
          "threshold was crossed; design smells are judged by structure. Pick non_smells only if "
          "the code is genuinely clean.\n\n"
        + 'End with the final answer as JSON on its own line: {"label": "<one_of_the_six>"}')
    return [{"role": "system", "content": SYSTEM_V4},
            {"role": "user", "content": user}]

def parse_label_v4(resp):
    for m in reversed(re.findall(r"\{[^{}]*\}", resp, re.S)):
        try:
            lab = str(json.loads(m).get("label", "")).strip().lower()
            if lab in CLASSES: return lab, True
        except Exception: pass
    low = resp.lower(); best_pos, best = -1, None
    for c in CLASSES:
        i = low.rfind(c)
        if i > best_pos: best_pos, best = i, c
    return (best, True) if best else ("non_smells", False)

MAX_CODE_CHARS = 8000

@torch.no_grad()
def classify_v4(code_text, language, max_new_tokens=400):
    code_text = code_text[:MAX_CODE_CHARS]
    def _run(ct, mlen):
        text = tok.apply_chat_template(build_messages_v4(ct, language),
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=mlen).to(model.device)
        try:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            del inputs; gc.collect(); torch.cuda.empty_cache()
    try:
        return _run(code_text, 3584)
    except Exception as e:
        if "out of memory" not in str(e).lower(): raise
        gc.collect(); torch.cuda.empty_cache()
        try:    return _run(code_text[:2500], 2560)
        except Exception as e2:
            gc.collect(); torch.cuda.empty_cache()
            return f"ERROR_OOM: {e2}"

In [ ]:
run_loop(classify_v4, parse_label_v4, "qwen_v4_predictions.csv")

[qwen_v4_predictions.csv] sudah selesai: 463 dari 463


qwen_v4_predictions.csv:   0%|          | 0/463 [00:00<?, ?it/s]

[qwen_v4_predictions.csv] SELESAI 463 sampel dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/qwen/qwen_v4_predictions.csv


In [ ]:
# =====================================================================
# TAMBAHAN — Varian Justifikasi untuk Telaah Interpretability
# Jalankan SETELAH model dimuat (setelah Cell 10) dan setelah `gt`,
# `read_code`, `RESULTS`, `CLASSES`, `tok`, `model` terdefinisi.
# Tujuan: membangkitkan ULANG penalaran naratif Qwen (yang sebelumnya
# tidak tersimpan) pada SAMPEL terstratifikasi, lalu menyimpan teks
# justifikasi PENUH (tanpa dipotong) ke qwen_justifications.csv.
# =====================================================================
import json, re, gc, os, time, random
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- Cek prasyarat: pastikan cell-cell sebelumnya sudah dijalankan di sesi ini ----
_perlu = {
    "RESULTS":   "Cell 5  (Hubungkan Google Drive)",
    "gt":        "Cell 7 & 8 (Unggah & ekstrak dataset -> membuat gt)",
    "read_code": "Cell 8  (definisi read_code)",
    "model":     "Cell 10 (Muat model Qwen)",
    "tok":       "Cell 10 (Muat tokenizer Qwen)",
    "CLASSES":   "Cell 12 (Definisi kelas)",
}
# _hilang = [f"  - {nama}  <- jalankan {asal}" for nama, asal in _perlu.items() if nama not in dir()]
_hilang = [f"  - {nama}  <- jalankan {asal}" for nama, asal in _perlu.items() if nama not in globals()]
if _hilang:
    raise RuntimeError(
        "Beberapa variabel/objek belum ada di sesi ini. Jalankan dulu cell berikut "
        "(berurutan dari atas), baru jalankan cell justifikasi ini:\n"
        + "\n".join(_hilang)
        + "\n\nTips: menu Colab > Runtime > Run before, lalu jalankan cell ini."
    )

PER_CLASS_PER_LANG = 5   # sampel per (kelas x bahasa) -> maks ~10/kelas, ~60 total, seimbang Java/Python
SEED               = 42
MAX_NEW_TOKENS = 384
MAX_CODE_CHARS = 8000
OUT_NAME       = "qwen_justifications.csv"

# ---- Definisi + instruksi yang MEMINTA penalaran (tanpa "JSON only, no explanation") ----
DEFINISI_JUSTIFY = """You are given ONE code unit (a single method/function, or a single class).
Classify it into EXACTLY ONE of these six categories, and EXPLAIN your reasoning:

1. long_method         - a single method/function with MORE THAN 50 lines of code.
2. god_class           - a class that concentrates many responsibilities (many methods/fields, low cohesion).
3. feature_envy        - a method that uses another class's data/methods more than its own.
4. data_class          - a class that only holds data (fields + getters/setters), with little/no behavior.
5. long_parameter_list - a method/function with MORE THAN 7 parameters.
6. non_smells          - clean code exhibiting none of the smells above.

The unit contains AT MOST ONE smell. Choose exactly one label."""

SYSTEM_JUSTIFY = ("You are an expert software engineer specialized in detecting code smells. "
                  "You explain your reasoning with concrete evidence, then give one final label.")

def build_messages_justify(code_text, language):
    user = (DEFINISI_JUSTIFY
        + f"\n\nLanguage: {language}\nCode unit:\n```{language}\n{code_text}\n```\n\n"
        + "Reason briefly but concretely. Cite specific evidence from the code, for example: "
          "the measured line count or parameter count; whether it is a class or a method; "
          "which external object/class it uses most (for feature_envy); or whether the class "
          "has real behavior or only getters/setters (for god_class vs data_class).\n\n"
        + "Reply with ONE JSON object on a single line, in exactly this form:\n"
        + '{\"reasoning\": \"<2-4 sentences citing concrete evidence>\", \"label\": \"<one_of_the_six>\"}')
    return [{"role": "system", "content": SYSTEM_JUSTIFY},
            {"role": "user", "content": user}]

def parse_justify(resp):
    """Ambil JSON terakhir yang valid; kembalikan (reasoning, label, parsed_ok)."""
    for m in reversed(re.findall(r"\{.*?\}", resp, re.S)):
        try:
            obj = json.loads(m)
            lab = str(obj.get("label", "")).strip().lower()
            rea = str(obj.get("reasoning", "")).strip()
            if lab in CLASSES:
                return (rea if rea else resp.strip()), lab, True
        except Exception:
            pass
    # fallback: label terakhir yang disebut, reasoning = seluruh teks
    low = resp.lower(); best_pos, best = -1, None
    for c in CLASSES:
        i = low.rfind(c)
        if i > best_pos: best_pos, best = i, c
    return resp.strip(), (best or "non_smells"), False

@torch.no_grad()
def justify(code_text, language, max_new_tokens=MAX_NEW_TOKENS):
    code_text = code_text[:MAX_CODE_CHARS]
    def _run(ct, mlen):
        text = tok.apply_chat_template(build_messages_justify(ct, language),
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=mlen).to(model.device)
        try:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            del inputs; gc.collect(); torch.cuda.empty_cache()
    try:
        return _run(code_text, 3072)
    except Exception as e:
        if "out of memory" not in str(e).lower(): raise
        gc.collect(); torch.cuda.empty_cache()
        try:    return _run(code_text[:2500], 2048)
        except Exception as e2:
            gc.collect(); torch.cuda.empty_cache()
            return f'{{"reasoning": "ERROR_OOM: {e2}", "label": "non_smells"}}'

# ---- Sampel terstratifikasi per (kelas x bahasa), deterministik ----
random.seed(SEED)
LANGS = sorted(gt["language"].unique())          # biasanya ['java', 'python']
sample_rows = []
for c in CLASSES:
    for lang in LANGS:
        sub = gt[(gt["label"] == c) & (gt["language"] == lang)]
        idx = list(sub.index)
        random.shuffle(idx)
        sample_rows += idx[:PER_CLASS_PER_LANG]   # ambil sebanyak yang tersedia (bisa < 5 utk sel kecil)
sample = gt.loc[sample_rows].reset_index(drop=True)
print("Distribusi sampel justifikasi (kelas x bahasa):")
print(pd.crosstab(sample["label"], sample["language"]))
print("Total sampel:", len(sample))

# ---- Jalankan (bisa dilanjutkan jika terputus) ----
out_path = os.path.join(RESULTS, OUT_NAME)
rows = pd.read_csv(out_path).to_dict("records") if os.path.exists(out_path) else []
done = {str(r["id"]) for r in rows}
t0 = time.time()
for _, r in tqdm(sample.iterrows(), total=len(sample), desc="justifikasi"):
    if str(r["id"]) in done:
        continue
    try:
        raw = justify(read_code(r), r["language"])
        rea, lab, ok = parse_justify(raw)
    except Exception as e:
        raw, rea, lab, ok = f"ERROR: {e}", f"ERROR: {e}", "non_smells", False
    rows.append({
        "id": str(r["id"]), "language": r["language"], "true_label": r["label"],
        "qwen_pred": lab, "correct": bool(lab == r["label"]),
        "reasoning": rea,          # teks penalaran hasil parsing
        "raw_full": str(raw),      # respons mentah PENUH (tanpa dipotong)
    })
    done.add(str(r["id"]))
    if len(rows) % 5 == 0:
        pd.DataFrame(rows).to_csv(out_path, index=False)
pd.DataFrame(rows).to_csv(out_path, index=False)
print(f"SELESAI {len(rows)} justifikasi dalam {(time.time()-t0)/60:.1f} menit -> {out_path}")

# ---- Unduh ke komputer untuk dikirim balik ----
try:
    from google.colab import files
    files.download(out_path)
except Exception as e:
    print("Lewati unduh otomatis:", e)

Distribusi sampel justifikasi (kelas x bahasa):
language             java  python
label                            
data_class              5       5
feature_envy            5       5
god_class               5       5
long_method             5       5
long_parameter_list     5       5
non_smells              5       5
Total sampel: 60


justifikasi:   0%|          | 0/60 [00:00<?, ?it/s]

SELESAI 60 justifikasi dalam 0.0 menit -> /content/drive/MyDrive/code-smell-llm/results/qwen/qwen_justifications.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Langkah 6 — Bandingkan semua varian

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef
import pandas as pd, os

def ringkas(nama, fname):
    path = os.path.join(RESULTS, fname)
    if not os.path.exists(path):
        print(f"\n=== {nama} ===  (belum dijalankan: {fname})"); return
    d = pd.read_csv(path)
    print(f"\n=== {nama} ===  Macro-F1 %.3f | MCC %.3f | Akurasi %.3f | gagal urai %d" % (
        f1_score(d.true_label, d.qwen_pred, labels=CLASSES, average="macro"),
        matthews_corrcoef(d.true_label, d.qwen_pred),
        (d.true_label==d.qwen_pred).mean(), int((~d.parsed).sum())))
    for c in CLASSES:
        sub = d[d.true_label==c]
        print(f"   {c:<22}: recall {(sub.qwen_pred==c).mean():.3f}")

ringkas("v1 ZERO-SHOT", "qwen_predictions.csv")
ringkas("v2 CoT",       "qwen_cot_predictions.csv")
ringkas("v3 SEIMBANG",  "qwen_v3_predictions.csv")
ringkas("v4 FEW-SHOT",  "qwen_v4_predictions.csv")


=== v1 ZERO-SHOT ===  Macro-F1 0.367 | MCC 0.379 | Akurasi 0.430 | gagal urai 0
   long_method           : recall 0.060
   god_class             : recall 0.171
   feature_envy          : recall 0.000
   data_class            : recall 0.894
   long_parameter_list   : recall 0.548
   non_smells            : recall 0.893

=== v2 CoT ===  Macro-F1 0.438 | MCC 0.474 | Akurasi 0.533 | gagal urai 0
   long_method           : recall 0.429
   god_class             : recall 0.024
   feature_envy          : recall 0.000
   data_class            : recall 0.894
   long_parameter_list   : recall 0.940
   non_smells            : recall 0.845

=== v3 SEIMBANG ===  Macro-F1 0.532 | MCC 0.518 | Akurasi 0.585 | gagal urai 0
   long_method           : recall 0.262
   god_class             : recall 0.768
   feature_envy          : recall 0.016
   data_class            : recall 0.879
   long_parameter_list   : recall 0.917
   non_smells            : recall 0.595

=== v4 FEW-SHOT ===  Macro-F1 0.338 | MCC 0

## Langkah 7 — (Opsional) Salin hasil ke komputer
Hasil sudah permanen di Google Drive, jadi sel ini opsional. Jalankan bila ingin salinan lokal.

In [ ]:
from google.colab import files
import os
for f in ["qwen_predictions.csv", "qwen_cot_predictions.csv",
          "qwen_v3_predictions.csv", "qwen_v4_predictions.csv"]:
    p = os.path.join(RESULTS, f)
    if os.path.exists(p):
        files.download(p)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys, importlib
print("Python :", sys.version.split()[0])
paket = ["torch", "transformers", "accelerate", "bitsandbytes",
         "sklearn", "scipy", "statsmodels", "pandas", "numpy",
         "matplotlib", "seaborn", "matplotlib_venn"]
for p in paket:
    try:
        m = importlib.import_module(p)
        print(f"{p:<16}: {m.__version__}")
    except Exception as e:
        print(f"{p:<16}: tidak terpasang ({e.__class__.__name__})")
import torch
print("CUDA   :", torch.version.cuda)

Python : 3.13.16
torch           : 2.11.0+cu130
transformers    : 5.22.1
accelerate      : 1.15.0
bitsandbytes    : 0.50.2
sklearn         : 1.6.1
scipy           : 1.16.3
statsmodels     : 0.15.0
pandas          : 2.2.3
numpy           : 2.4.6
matplotlib      : 3.10.0
seaborn         : 0.13.2
matplotlib_venn : 1.1.2
CUDA   : 13.0


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
import torch, transformers, accelerate, bitsandbytes
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

torch       : 2.11.0+cu130
transformers: 5.22.1
accelerate  : 1.15.0
bitsandbytes: 0.50.2
